# Iterative Self-Play + Train Loop

Notebook này chạy nhiều iteration theo chuỗi: sinh self-play mới -> train checkpoint mới -> dùng checkpoint đó cho iteration tiếp theo.

In [ ]:
from pathlib import Path
import shlex
import subprocess


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "legacy" / "AlphaZero-based-autonomous-driving").is_dir() and (candidate / "source" / "highway-env").is_dir():
            return candidate
    raise RuntimeError("Could not locate repo root. Open the notebook from inside this repository.")


def run_command(command, cwd: Path):
    print("$ " + " ".join(shlex.quote(str(part)) for part in command))
    process = subprocess.Popen(
        [str(part) for part in command],
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}")


repo_root = find_repo_root()
package_root = repo_root / "legacy" / "AlphaZero-based-autonomous-driving"
runner = ["uv", "run", "python", "-m"]
self_play_module = "AlphaZero.scripts.self_play_parallel"
train_module = "AlphaZero.scripts.train_from_self_play"
print(f"repo_root={repo_root}")
print(f"package_root={package_root}")
print(f"self_play_module={self_play_module}")
print(f"train_module={train_module}")

In [ ]:
run_name = "notebook_loop_run"
num_iterations = 3
loop_root = repo_root / "legacy" / "AlphaZero-based-autonomous-driving" / "outputs" / "self_play_train_loop" / run_name

self_play_config = {
    "workers": 2,
    "episodes_per_worker": 2,
    "duration": 80,
    "other_vehicles": 1,
    "finish_laps": 1,
    "terminate_on_finish": True,
    "n_simulations": 5,
    "c_puct": 2.5,
    "temperature": 1.0,
    "temperature_drop_step": None,
    "dirichlet_alpha": 0.3,
    "root_exploration_fraction": 0.25,
    "n_residual_layers": 10,
    "device": "auto",
    "torch_threads_per_worker": 1,
    "progress_interval": 5,
    "max_steps_per_episode": None,
}

train_config = {
    "pattern": "*.pt",
    "recursive": False,
    "limit_files": None,
    "n_residual_layers": self_play_config["n_residual_layers"],
    "batch_size": 32,
    "epochs": 10,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "device": "auto",
    "shuffle": True,
}

loop_root

In [ ]:
loop_root.mkdir(parents=True, exist_ok=True)
current_model = None
iteration_artifacts = []

for iteration in range(num_iterations):
    iteration_dir = loop_root / f"iteration_{iteration:03d}"
    self_play_dir = iteration_dir / "self_play"
    model_out = iteration_dir / "model.pth"
    metrics_out = iteration_dir / "model.pth.metrics.json"
    iteration_dir.mkdir(parents=True, exist_ok=True)

    self_play_command = [
        *runner,
        self_play_module,
        "--workers", str(self_play_config["workers"]),
        "--episodes-per-worker", str(self_play_config["episodes_per_worker"]),
        "--duration", str(self_play_config["duration"]),
        "--other-vehicles", str(self_play_config["other_vehicles"]),
        "--finish-laps", str(self_play_config["finish_laps"]),
        "--n-simulations", str(self_play_config["n_simulations"]),
        "--c-puct", str(self_play_config["c_puct"]),
        "--temperature", str(self_play_config["temperature"]),
        "--n-residual-layers", str(self_play_config["n_residual_layers"]),
        "--device", str(self_play_config["device"]),
        "--torch-threads-per-worker", str(self_play_config["torch_threads_per_worker"]),
        "--progress-interval", str(self_play_config["progress_interval"]),
        "--output-dir", str(self_play_dir),
    ]
    self_play_command.append("--terminate-on-finish" if self_play_config["terminate_on_finish"] else "--no-terminate-on-finish")
    if self_play_config["temperature_drop_step"] is not None:
        self_play_command.extend(["--temperature-drop-step", str(self_play_config["temperature_drop_step"])])
    if self_play_config["dirichlet_alpha"] is not None:
        self_play_command.extend(["--dirichlet-alpha", str(self_play_config["dirichlet_alpha"])])
    if self_play_config["root_exploration_fraction"] is not None:
        self_play_command.extend(["--root-exploration-fraction", str(self_play_config["root_exploration_fraction"])])
    if self_play_config["max_steps_per_episode"] is not None:
        self_play_command.extend(["--max-steps-per-episode", str(self_play_config["max_steps_per_episode"])])
    if current_model is not None:
        self_play_command.extend(["--model-path", str(current_model)])

    print(f"\n=== iteration {iteration} :: self-play ===")
    run_command(self_play_command, cwd=package_root)

    train_command = [
        *runner,
        train_module,
        "--input-dir", str(self_play_dir),
        "--pattern", str(train_config["pattern"]),
        "--model-out", str(model_out),
        "--metrics-out", str(metrics_out),
        "--n-residual-layers", str(train_config["n_residual_layers"]),
        "--batch-size", str(train_config["batch_size"]),
        "--epochs", str(train_config["epochs"]),
        "--learning-rate", str(train_config["learning_rate"]),
        "--weight-decay", str(train_config["weight_decay"]),
        "--device", str(train_config["device"]),
    ]
    if train_config["recursive"]:
        train_command.append("--recursive")
    if train_config["limit_files"] is not None:
        train_command.extend(["--limit-files", str(train_config["limit_files"])])
    if current_model is not None:
        train_command.extend(["--model-in", str(current_model)])
    if not train_config["shuffle"]:
        train_command.append("--no-shuffle")

    print(f"\n=== iteration {iteration} :: train ===")
    run_command(train_command, cwd=package_root)

    current_model = model_out
    artifacts = {
        "iteration": iteration,
        "self_play_dir": self_play_dir,
        "model_out": model_out,
        "metrics_out": metrics_out,
    }
    iteration_artifacts.append(artifacts)
    print(f"\ncompleted iteration {iteration}: {artifacts}")

iteration_artifacts